# Example: Ticker-Picker Bandit (Advanced, Optional)

This is an **advanced optional exercise** that extends the bandit framework from Example 2 to a harder problem: **asset selection**. Instead of learning which elasticity η works best, the bandit learns which subset of assets to include in the portfolio. With $K$ assets, there are $2^K - 1$ possible non-empty subsets (arms). The Cobb-Douglas utility over the included assets serves as the reward.

The earlier S3 examples kept the asset universe fixed and tuned how to weight it. This example asks the harder question: should some assets be dropped entirely? A bandit over $2^K - 1$ subsets lets the data answer, but the trade is concentrating on the strongest alphas versus giving up the diversification that even weak assets still provide.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
> * __Run the combinatorial bandit on a single path:__ Train an epsilon-greedy bandit over all non-empty asset subsets and visualize reward convergence, exploration decay, and the best-discovered subset. Read the convergence plot to judge whether the iteration budget was sufficient.
> * __Backtest the bandit-selected portfolio across Monte Carlo paths:__ Run the bandit selector on each path, apply Cobb-Douglas allocation to the selected assets, and compare against the full-universe engine and an equal-weight buy-and-hold. Use the median scorecard and box plot to see whether asset selection adds value at the distributional level.
> * __Evaluate the ticker-picker against the validation gates:__ Apply the same pass/fail criteria from Example 3 to determine whether the bandit-selected portfolio is deployment-ready. The benchmark wealth threshold is the equal-weight buy-and-hold, not the frozen Cobb-Douglas baseline.

This is a combinatorially harder problem than elasticity learning: the action space grows exponentially with the number of assets.

___

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via [the `Include.jl` file](./Include.jl). This activates the local [Julia](https://julialang.org) environment and loads all dependencies.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

### Constants
In the section below, we set some constants that will be used throughout the notebook. We can modify these constants to explore different scenarios and allocations.

See the comments in the code for more details on each constant, its purpose, units, etc.

In [ ]:
# Ticker-picker configuration
B₀ = 10_000.0                       # starting budget (USD)
Δt = 1.0 / 252.0                    # trading-day step (years)
L_short = 21                        # short EMA window (days) for the sentiment crossover
L_long = 63                         # long EMA window (days) for the sentiment crossover
L_growth = 10                       # EMA window (days) for smoothed market growth rate
GAIN = 10.0                         # gain constant G for the λ sentiment signal (dimensionless)
offset = L_short + L_long           # warmup offset before trading begins (days)
SCENARIO_SEED = 2026                # RNG seed for the hybrid-SIM scenario (locked across Examples 1-2)
SCENARIO_PATHS = 500                # number of Monte Carlo paths in the bandit backtest
SCENARIO_ACTIVE_DAYS = 252          # trading horizon after warmup (days)
BANDIT_EPSILON = 0.1                # initial exploration probability for the epsilon-greedy bandit
BANDIT_ALPHA = 0.1                  # constant learning rate for bandit arm-mean updates
BANDIT_ITERS_SINGLE = 500           # bandit iteration budget for the single-path Task 1 run
BANDIT_ITERS_BACKTEST = 200         # bandit iteration budget per path inside the Monte Carlo backtest
TRIGGER_MAX_DRAWDOWN = 0.15         # drawdown trigger threshold (de-risks to cash, fraction)
TRIGGER_MAX_TURNOVER = 0.50         # max fraction of wealth traded per rebalance
ENGINE_PRIOR_CCGR_PCT = 8.0         # bias-correction anchor (% /yr); matches Session 2 + EWLS notebook (applied to the two daily-rebalanced arms only)

# --- Validation gates (shared with Example 3, applied in Task 3) ---
VALIDATION_MIN_SHARPE = 0.3         # minimum acceptable median Sharpe ratio
VALIDATION_MAX_DRAWDOWN = 0.25      # maximum acceptable median drawdown (fraction)
VALIDATION_MAX_FAILURE_RATE = 0.10  # max acceptable fraction of paths ending below B₀

Load Session 1 artifacts and regenerate the scenario with the same seed as Examples 1-2.

In the code block below, we load the Session 1 minimum-variance pack, build the SIM-parameter dictionary keyed by ticker, instantiate the [`MyMarketSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyMarketSurrogateModel) and [`MyPortfolioSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyPortfolioSurrogateModel) calibrated in Session 1, and call [`generate_hybrid_scenario(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.generate_hybrid_scenario) with the locked seed.

The cell returns:

* `my_tickers::Vector{String}`: Session 1 ticker universe (same order as `allocation_weights`).
* `sim_estimates::Vector{MySIMParameterEstimate}`: per-ticker SIM fits with α, β, σ_ε.
* `sim_params::Dict{String,Tuple{Float64,Float64,Float64}}`: allocator-adapter view of `sim_estimates`, keyed by ticker.
* `g_f::Float64`: continuously compounded annual risk-free rate (1/yr); used to discount terminal wealth into NPV against the risk-free baseline.
* [`market_model::MyMarketSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyMarketSurrogateModel): 50-state generative model market surrogate fit to SPY.
* [`portfolio::MyPortfolioSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyPortfolioSurrogateModel): per-ticker generative marginals and Student-t copula for path composition.
* `calib::Dict{String,Any}`: full SIM calibration dict (source of truth for α, β, σ_ε, and bootstrap SEs).
* `start_prices::Dict{String,Float64}`: per-ticker starting prices for forward scenarios.
* `N::Int`: number of tickers in the universe.
* `scenario::MyBacktestScenario`: hybrid forward path ensemble (seed-locked for reproducibility across Examples 1-2).

In [ ]:
(; my_tickers, sim_estimates, sim_params, g_f,
   market_model, portfolio, calib, start_prices, N, scenario) = let
    # --- Step 1: Load S1 artifacts ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    my_tickers    = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    g_f           = haskey(minvar, "g_f") ? Float64(minvar["g_f"]) : Float64(minvar["r_f"]);
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        e.ticker => (e.α, e.β, e.σ_ε) for e in sim_estimates
    );

    # --- Step 2: Surrogates and scenario ---
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = load_results(joinpath(_PATH_TO_DATA_S1, "sim-parameter-estimates.jld2"));
    snap        = MyCurrentPrices();
    snap_lookup = Dict(snap["tickers"] .=> snap["prices"]);
    start_prices = Dict(t => snap_lookup[t] for t in my_tickers);

    # --- Step 3: Dimensions ---
    N         = length(my_tickers);
    n_trading_days   = SCENARIO_ACTIVE_DAYS;
    T_total          = offset + n_trading_days;

    # --- Step 4: Regenerate scenario ---
    Random.seed!(SCENARIO_SEED);
    scenario = generate_hybrid_scenario(
        market_model, portfolio, calib, my_tickers;
        n_paths = SCENARIO_PATHS, n_steps = T_total,
        start_prices = start_prices, label = "S3 Ticker-Picker", seed = SCENARIO_SEED);

    println("Loaded $(N) tickers ($(2^N - 1) possible subsets), $(scenario.n_paths) paths")
    (my_tickers = my_tickers, sim_estimates = sim_estimates, sim_params = sim_params,
     g_f = g_f, market_model = market_model, portfolio = portfolio, calib = calib,
     start_prices = start_prices, N = N, scenario = scenario)
end

___
## Task 1: Run the Combinatorial Bandit on a Single Path
In this task, we extract the first path, build the bandit context at the offset day, and run the epsilon-greedy solver over all $2^K - 1$ arms. Each arm is a binary vector indicating which assets to include, and [`solve_bandit(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.solve_bandit) returns the arm pull counts, reward trace, and best-subset action.

> __What should we see?__
>
> The reward trace should converge as the bandit discovers the best asset subset. The best action should reflect which tickers have the strongest preference weights at the given market state.

The code block below returns `bandit_result::`[`MyBanditResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBanditResult) carrying the arm means, pull counts, reward trace, and best-subset arm for the $2^K - 1$-arm combinatorial bandit.

In [ ]:
bandit_result = let
    # --- Step 1: Extract first path at offset ---
    p_idx = 1;
    mkt = scenario.market_paths[p_idx, :];
    ema_s = compute_ema(mkt; window = L_short);
    ema_l = compute_ema(mkt; window = L_long);
    \u03bb = compute_lambda(ema_s, ema_l; G = GAIN);
    gm_raw = compute_market_growth(mkt; \u0394t = \u0394t);
    gm_e = compute_ema(gm_raw; window = L_growth);

    prices_at_offset = [scenario.price_paths[p_idx, offset, k] for k in 1:N];

    # --- Step 2: Build bandit context ---
    bandit_ctx = build(MyBanditContext, (
        tickers = my_tickers, sim_parameters = sim_params,
        prices = prices_at_offset, B = B\u2080,
        gm_t = gm_e[min(offset, length(gm_e))],
        lambda = \u03bb[offset], epsilon = BANDIT_EPSILON
    ));

    # --- Step 3: Run bandit ---
    bandit_model = build(MyEpsilonGreedyBanditModel, (
        K = N, n_iterations = BANDIT_ITERS_SINGLE, alpha = BANDIT_ALPHA
    ));
    bandit_result = solve_bandit(bandit_model, bandit_ctx);

    # --- Step 4: Report ---
    selected = bandit_result.best_action;
    selected_tickers = my_tickers[selected .== 1];
    println("Best subset ($(length(selected_tickers)) of $(N) assets):")
    println("  $(selected_tickers)")
    println("  Utility: $(round(bandit_result.best_utility, digits=4))")

    # --- Step 5: Visualize convergence ---
    p1 = plot(bandit_result.reward_history, alpha=0.3, color=:gray50, label="Reward",
        ylabel="CD Utility", title="Ticker-Picker: Reward Trace");
    p2 = plot(bandit_result.exploration_history, color=:coral, lw=2, label="\u03b5_t",
        ylabel="\u03b5", xlabel="Iteration", title="Exploration Decay");
    regret = compute_regret(bandit_result.reward_history);
    p3 = plot(regret, color=:firebrick, lw=2, label="",
        ylabel="Cumulative Regret", xlabel="Iteration", title="Regret");

    display(plot(p1, p2, p3, layout=(3,1), size=(800, 600)))
    bandit_result
end


___
## Task 2: Backtest Bandit-Selected Portfolios Across All Paths
In this task, we run [the `backtest_bandit(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_bandit) across all paths. On each path, the bandit selects an asset subset at the offset day, then the Cobb-Douglas engine runs with only the selected assets for the remainder of the trading horizon.

> __What should we see?__
>
> The bandit-selected portfolio should focus on the highest-alpha assets. Whether this beats the full-universe engine depends on whether excluding low-quality assets helps more than the diversification benefit they provide. Both the ticker-picker and the full-universe Cobb-Douglas baseline are daily-rebalanced and inherit the SIM rebalancing-alpha artifact that Session 2 documented; we apply the same `ENGINE_PRIOR_CCGR_PCT` bias-correction anchor to both so the median levels are comparable. The buy-and-hold benchmark is passive and does not carry the artifact, so it stays at its raw synthetic level.

The code block below returns the three bias-treated per-path backtest results: `ticker_bt::`[`MyBacktestResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBacktestResult) (bandit-selected subset engine, bias-corrected), `engine_bt::MyBacktestResult` (full-universe Cobb-Douglas baseline via [`backtest_engine`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_engine), bias-corrected), and `bh_bt::MyBacktestResult` (equal-weight buy-and-hold benchmark via [`backtest_buyhold`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_buyhold), raw); plus three corresponding NPV vectors `ticker_npv::Vector{Float64}`, `engine_npv::Vector{Float64}`, `bh_npv::Vector{Float64}`.

In [ ]:
(; ticker_bt, engine_bt, bh_bt, ticker_npv, engine_npv, bh_npv) = let
    # --- Step 0: Local bias-correction helper (mirrors the EWLS + BanditEta notebooks) ---
    function _apply_bias(r::MyBacktestResult, bias_pct_per_yr::Float64, Δt::Float64)::MyBacktestResult
        n_t, n_p = size(r.wealth_paths);
        drag = exp.(-bias_pct_per_yr / 100.0 .* (0:(n_t-1)) .* Δt);
        Wc = r.wealth_paths .* drag;
        final_wealth_c  = Wc[end, :];
        max_drawdowns_c = zeros(n_p);
        sharpe_ratios_c = zeros(n_p);
        for p ∈ 1:n_p
            wealth = Wc[:, p];
            peak = accumulate(max, wealth);
            max_drawdowns_c[p] = maximum((peak .- wealth) ./ peak);
            vol = std(diff(wealth) ./ wealth[1:end-1]) * sqrt(252);
            mean_ret = (wealth[end] / wealth[1] - 1.0);
            sharpe_ratios_c[p] = vol > 1e-6 ? mean_ret / vol : 0.0;
        end
        rc = MyBacktestResult();
        rc.scenario_label = r.scenario_label;
        rc.strategy_label = r.strategy_label * " [bias-corrected $(round(bias_pct_per_yr, digits=2))pp/yr]";
        rc.final_wealth   = final_wealth_c;
        rc.max_drawdowns  = max_drawdowns_c;
        rc.sharpe_ratios  = sharpe_ratios_c;
        rc.wealth_paths   = Wc;
        return rc;
    end

    # --- Step 1: Run bandit backtest (daily rebalancing on selected subset) ---
    ticker_bt_raw = backtest_bandit(scenario, my_tickers, sim_params;
        B₀ = B₀, offset = offset, N_short = L_short, N_long = L_long,
        GAIN = GAIN, N_growth = L_growth, n_bandit_iters = BANDIT_ITERS_BACKTEST);
    ticker_bt_raw.strategy_label = "Ticker-Picker raw";

    # --- Step 2: Run full-universe engine for comparison (daily rebalancing) ---
    rules_params = (max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER);
    engine_bt_raw = backtest_engine(scenario, my_tickers, sim_params, rules_params;
        B₀ = B₀, offset = offset);
    engine_bt_raw.strategy_label = "Full-Universe CD raw";

    # --- Step 3: Run buy-and-hold (passive; no bias correction needed) ---
    bh_bt = backtest_buyhold(scenario, my_tickers; B₀ = B₀, offset = offset);

    # --- Step 4: Bias correction anchored to ENGINE_PRIOR_CCGR_PCT, applied to the two daily-rebalanced arms only ---
    ticker_raw_med_ccgr = let
        gs = Float64[];
        n_paths = size(ticker_bt_raw.wealth_paths, 2);
        for p ∈ 1:n_paths
            w = ticker_bt_raw.wealth_paths[:, p];
            push!(gs, log(w[end] / w[1]) / ((length(w) - 1) * Δt) * 100);
        end
        median(gs)
    end;
    bias_pct_per_yr = ticker_raw_med_ccgr - ENGINE_PRIOR_CCGR_PCT;
    println("  Ticker-Picker raw median CCGR: $(round(ticker_raw_med_ccgr, digits=2)) %/yr");
    println("  ENGINE_PRIOR_CCGR_PCT:         $(ENGINE_PRIOR_CCGR_PCT) %/yr");
    println("  Bias correction drag:          $(round(bias_pct_per_yr, digits=2)) pp/yr (applied to the two daily-rebalanced arms only; BH stays raw)");

    ticker_bt = _apply_bias(ticker_bt_raw, bias_pct_per_yr, Δt);
    engine_bt = _apply_bias(engine_bt_raw, bias_pct_per_yr, Δt);

    # --- Step 5: Per-path NPV against the continuously-compounded risk-free baseline ---
    # NPV_p = W_T,p · exp(-g_f · T_active) − B₀.  Positive ⇒ path beat risk-free in today's dollars.
    T_active   = SCENARIO_ACTIVE_DAYS * Δt;
    discount   = exp(-g_f * T_active);
    ticker_npv = ticker_bt.final_wealth .* discount .- B₀;
    engine_npv = engine_bt.final_wealth .* discount .- B₀;
    bh_npv     = bh_bt.final_wealth     .* discount .- B₀;

    # --- Step 6: Wide-form scorecard (one row per strategy) ---
    strategies_pairs = [
        ("Ticker-Picker (corrected)",      ticker_bt, ticker_npv),
        ("Full-Universe CD (corrected)",   engine_bt, engine_npv),
        ("Buy-and-Hold (raw)",             bh_bt,     bh_npv),
    ];
    scorecard_df = DataFrame(
        "Strategy"             => [s for (s, _, _) in strategies_pairs],
        "Median W/W₀"          => [round(median(bt.final_wealth) / B₀, digits = 3) for (_, bt, _) in strategies_pairs],
        "Median Max DD (%)"    => [round(median(bt.max_drawdowns) * 100, digits = 1) for (_, bt, _) in strategies_pairs],
        "Median Sharpe"        => [round(median(bt.sharpe_ratios), digits = 3) for (_, bt, _) in strategies_pairs],
        "Fail rate (%)"        => [round(mean(bt.final_wealth .< B₀) * 100, digits = 1) for (_, bt, _) in strategies_pairs],
        "Median NPV (\$)"      => [round(median(npv), digits = 0) for (_, _, npv) in strategies_pairs],
        "Median NPV (% of B₀)" => [round(median(npv) / B₀ * 100, digits = 2) for (_, _, npv) in strategies_pairs],
        "P[NPV<0] (%)"         => [round(mean(npv .< 0) * 100, digits = 1) for (_, _, npv) in strategies_pairs],
    );
    println("Ticker-picker backtest: $(scenario.n_paths) paths (engine arms bias-corrected to anchor=$(ENGINE_PRIOR_CCGR_PCT) %/yr)")
    pretty_table(scorecard_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Panel 1: Terminal-wealth box plot ---
    labels = ["Ticker-Picker" "Full-Universe" "Buy-and-Hold"];
    wealth_data = hcat(ticker_bt.final_wealth ./ B₀, engine_bt.final_wealth ./ B₀, bh_bt.final_wealth ./ B₀);
    p = boxplot(labels, wealth_data, legend=false, ylabel="W/W₀",
        title="Ticker-Picker vs Full-Universe vs Buy-and-Hold", color=:steelblue, alpha=0.7);
    hline!(p, [1.0], linestyle=:dash, color=:red, label="");
    display(p)
    (ticker_bt = ticker_bt, engine_bt = engine_bt, bh_bt = bh_bt,
     ticker_npv = ticker_npv, engine_npv = engine_npv, bh_npv = bh_npv)
end

___
## Task 3: Validation Report for the Ticker-Picker
In this task, we apply the same five deployment criteria from Example 3 to the ticker-picker strategy, with one substitution: the wealth gate is measured against a real equal-weight buy-and-hold here rather than the frozen Cobb-Douglas baseline used in Example 3. Buy-and-hold is the benchmark, not a candidate, so only Ticker-Picker and the Full-Universe CD engine are evaluated against the gates.

> __What should we see?__
>
> The ticker-picker may pass or fail depending on whether asset exclusion helps. In a 10-asset universe, excluding the weakest assets can improve risk-adjusted returns. In smaller universes, the loss of diversification may dominate.

In the code block below, we build the five-gate criteria dictionary, loop over the two candidate strategies, and print a per-strategy pass/fail table using [`MyValidationReport`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyValidationReport).

In [ ]:
let
    # --- Step 1: Build validation criteria dict (thresholds shared with Example 3, BH benchmark + NPV gate) ---
    bh_median_wealth = median(bh_bt.final_wealth);

    criteria = Dict(
        "min_sharpe" => VALIDATION_MIN_SHARPE,
        "max_drawdown" => VALIDATION_MAX_DRAWDOWN,
        "max_failure_rate" => VALIDATION_MAX_FAILURE_RATE,
        "min_wealth_vs_bh" => bh_median_wealth,
        "min_median_npv" => 0.0,
    );

    # --- Step 2: Candidate strategies (BH is the benchmark, not a candidate) ---
    strategies = [
        ("Ticker-Picker Bandit",    ticker_bt, ticker_npv),
        ("Full-Universe CD Engine", engine_bt, engine_npv),
    ];

    # --- Step 3: Build per-strategy validation reports ---
    reports = MyValidationReport[];
    for (label, bt, npv) in strategies
        actuals = Dict(
            "min_sharpe" => median(bt.sharpe_ratios),
            "max_drawdown" => median(bt.max_drawdowns),
            "max_failure_rate" => mean(bt.final_wealth .< B₀),
            "min_wealth_vs_bh" => median(bt.final_wealth),
            "min_median_npv" => median(npv),
        );
        push!(reports, build(MyValidationReport;
            strategy_label = label, criteria = criteria, actuals = actuals));
    end

    # --- Step 4: Wide-form pass/fail DataFrame (one row per strategy) ---
    fmt_cell = (r, k, op) -> "$(r.passed[k] ? "✓" : "✗") ($(round(r.actuals[k], digits=3)) $(op) $(round(r.criteria[k], digits=3)))"

    report_df = DataFrame(
        "Strategy"      => [r.strategy_label for r in reports],
        "Sharpe"        => [fmt_cell(r, "min_sharpe", "≥") for r in reports],
        "Drawdown"      => [fmt_cell(r, "max_drawdown", "≤") for r in reports],
        "Failure rate"  => [fmt_cell(r, "max_failure_rate", "≤") for r in reports],
        "Wealth vs BH"  => [fmt_cell(r, "min_wealth_vs_bh", "≥") for r in reports],
        "NPV vs RF"     => [fmt_cell(r, "min_median_npv", "≥") for r in reports],
        "Verdict"       => [all(values(r.passed)) ? "PASS" : "FAIL" for r in reports],
    );
    println("Ticker-picker validation report (BH wealth benchmark + NPV-vs-RF gate)")
    pretty_table(report_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end;

___
## Summary
This advanced example applied the combinatorial epsilon-greedy bandit to the asset selection problem, backtested across the Monte Carlo path ensemble with consistent bias treatment on the two daily-rebalanced arms, and evaluated against the same deployment gates used in Example 3.

> __Key Takeaways:__
>
> * __The action space is exponential:__ The number of non-empty asset subsets grows exponentially with the universe size, so epsilon-greedy converges only when the universe is small. For larger universes, this motivates more efficient exploration strategies such as Thompson sampling or contextual bandits.
> * __Asset exclusion trades diversification for concentration:__ Removing weak assets can improve risk-adjusted returns, but reducing the universe also reduces diversification. The net effect is path-dependent.
> * __The ticker-picker and elasticity bandit are complementary:__ One decides which assets to include; the other decides how aggressively to concentrate on them. Combining both is a natural next step.

The validation report in Task 3 provides the clearest read on whether the combinatorial search budget is large enough to justify the exponential action space; the same gates that govern the core engine apply unchanged to this asset-selection variant, so the pass/fail verdict transfers directly into the Session 4 compliance discussion.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.